In [56]:
import pandas as pd

df = pd.read_excel(r"G:\log&DT\Telco_customer_churn.xlsx")
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [57]:
# =========================
# 1. Import Libraries
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

In [58]:
# Shape
print("Shape:", df.shape)

# Data types
df.info()


Shape: (7043, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Intern

In [59]:

# Check missing values
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [60]:
import pandas as pd
import numpy as np

df = df.copy()

# Clean column names: lower + underscore
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

df.columns

Index(['customerid', 'count', 'country', 'state', 'city', 'zip_code',
       'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen',
       'partner', 'dependents', 'tenure_months', 'phone_service',
       'multiple_lines', 'internet_service', 'online_security',
       'online_backup', 'device_protection', 'tech_support', 'streaming_tv',
       'streaming_movies', 'contract', 'paperless_billing', 'payment_method',
       'monthly_charges', 'total_charges', 'churn_label', 'churn_value',
       'churn_score', 'cltv', 'churn_reason'],
      dtype='object')

In [61]:
# total_charges is object -> numeric
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")

# senior_citizen is object -> 0/1
# (sometimes it comes as "Yes/No" or "0/1" strings)
df["senior_citizen"] = df["senior_citizen"].replace({"Yes": 1, "No": 0})
df["senior_citizen"] = pd.to_numeric(df["senior_citizen"], errors="coerce")

C:\Users\baraa\AppData\Local\Temp\ipykernel_23140\1480891763.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["senior_citizen"] = df["senior_citizen"].replace({"Yes": 1, "No": 0})


In [62]:
print(df["total_charges"].isna().sum())
# usually missing because tenure_months = 0
df["total_charges"] = df["total_charges"].fillna(0)

11


In [63]:
TARGET = "churn_value"

leakage_cols = ["churn_score", "churn_reason", "cltv", "churn_label"]
id_cols = ["customerid", "count"]  # count is useless

drop_cols = [c for c in leakage_cols + id_cols if c in df.columns]
df = df.drop(columns=drop_cols)

df.shape

(7043, 27)

In [64]:
location_cols = ["country", "state", "city", "zip_code", "lat_long", "latitude", "longitude"]
drop_loc = [c for c in location_cols if c in df.columns]

df_no_loc = df.drop(columns=drop_loc)
df_no_loc.shape

(7043, 20)

In [65]:
work_df = df_no_loc.copy()

work_df["tenure_group"] = pd.cut(
    work_df["tenure_months"],
    bins=[-1, 0, 6, 12, 24, 48, 72],
    labels=["0", "1-6", "7-12", "13-24", "25-48", "49-72"]
)

In [66]:
work_df["avg_monthly_spend"] = np.where(
    work_df["tenure_months"] > 0,
    work_df["total_charges"] / work_df["tenure_months"],
    0
)

In [67]:
from sklearn.model_selection import train_test_split

y = work_df[TARGET].astype(int)
X = work_df.drop(columns=[TARGET])

# one-hot encode
X = pd.get_dummies(X, drop_first=True)

X.shape, y.shape

((7043, 36), (7043,))

In [68]:
X.shape

(7043, 36)

In [69]:
y.shape

(7043,)

In [70]:

# Split with stratify (important for churn imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [79]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

def eval_model(name, model, X_train, y_train, X_test, y_test, proba=True):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds ),
        "f1": f1_score(y_test, preds),
        "f1_macro": f1_score(y_test, preds, average="macro")

    }
    
    print(f"\n=== {name} ===")
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    print("\nClassification Report:\n", classification_report(y_test, preds))

    return metrics, model

In [ ]:
# Import required tools for building the model pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Pipeline = clean + transform + model in one safe workflow.

# Build a Pipeline:
# Pipeline = steps that will be applied sequentially during training and prediction
logreg_base = Pipeline(steps=[
    
    # Step 1: Scale the features
    # Logistic Regression is sensitive to feature scale.
    # We use StandardScaler to normalize features (mean=0, std=1).
    # with_mean=False is used because one-hot encoding can create sparse-like structure.
    ("scaler", StandardScaler(with_mean=False)),
    
    # Step 2: Logistic Regression model
    # max_iter=2000 increases training iterations to ensure convergence.
    ("clf", LogisticRegression(max_iter=2000))
])

# Train the model and evaluate it using our custom evaluation function
# This will:
# - Fit the model on training data
# - Predict on test data
# - Calculate accuracy, precision, recall, F1, etc.
m1, logreg_base = eval_model(
    "LogReg_Baseline",
    logreg_base,
    X_train,
    y_train,
    X_test,
    y_test
)


=== LogReg_Baseline ===
Confusion Matrix:
 [[926 109]
 [169 205]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.55      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [81]:
from sklearn.tree import DecisionTreeClassifier

tree_base = DecisionTreeClassifier(random_state=42)

m2, tree_base = eval_model("Tree_Baseline", tree_base, X_train, y_train, X_test, y_test, proba=True)


=== Tree_Baseline ===
Confusion Matrix:
 [[822 213]
 [196 178]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.79      0.80      1035
           1       0.46      0.48      0.47       374

    accuracy                           0.71      1409
   macro avg       0.63      0.64      0.63      1409
weighted avg       0.71      0.71      0.71      1409



In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_pipe = Pipeline(steps=[
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(max_iter=3000))
])

param_grid_logreg = {
    "clf__C": [0.01, 0.1, 1, 5, 10],
    "clf__penalty": ["l2"], # l1 , elasticnet , None
    "clf__solver": ["lbfgs", "liblinear"]
}

grid_logreg = GridSearchCV(
    estimator=logreg_pipe,
    param_grid=param_grid_logreg,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

grid_logreg.fit(X_train, y_train)
print("Best params:", grid_logreg.best_params_)

Best params: {'clf__C': 5, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}


The **solver** is the algorithm that Logistic Regression uses to find the best weights for the model.

- lbfgs: Limited-memory Broyden–Fletcher–Goldfarb–Shanno 
- liblinear: Library for Linear Classification 


| Solver    | Best for        | L1 Support | Multi-class |
| --------- | --------------- | ---------- | ----------- |
| lbfgs     | Larger datasets | No         | Yes         |
| liblinear | Small datasets  | Yes        | Limited     |


In [83]:
m3, best_logreg = eval_model(
    "LogReg_Tuned",
    grid_logreg.best_estimator_,
    X_train, y_train, X_test, y_test
)


=== LogReg_Tuned ===
Confusion Matrix:
 [[927 108]
 [170 204]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.65      0.55      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [84]:
tree = DecisionTreeClassifier(random_state=42)

param_grid_tree = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 10, 20, 50],
    "min_samples_leaf": [1, 5, 10, 20],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(
    estimator=tree,
    param_grid=param_grid_tree,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

grid_tree.fit(X_train, y_train)
print("Best params:", grid_tree.best_params_)

Best params: {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [85]:
m4, best_tree = eval_model(
    "Tree_Tuned",
    grid_tree.best_estimator_,
    X_train, y_train, X_test, y_test
)


=== Tree_Tuned ===
Confusion Matrix:
 [[836 199]
 [152 222]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.81      0.83      1035
           1       0.53      0.59      0.56       374

    accuracy                           0.75      1409
   macro avg       0.69      0.70      0.69      1409
weighted avg       0.76      0.75      0.76      1409



In [86]:
results = pd.DataFrame([m1, m2, m3, m4]).sort_values("f1", ascending=False)
results

,model,accuracy,precision,recall,f1,f1_macro
0,LogReg_Baseline,0.802697,0.652866,0.548128,0.595930,0.732707
2,LogReg_Tuned,0.802697,0.653846,0.545455,0.594752,0.732179
3,Tree_Tuned,0.750887,0.527316,0.593583,0.558491,0.692493
1,Tree_Baseline,0.709723,0.455243,0.475936,0.465359,0.633069


In [90]:
# Get probabilities for class 1 (Churn)
probs = best_logreg.predict_proba(X_test)[:, 1]
probs

array([0.07326654, 0.67985506, 0.0881734 , ..., 0.2083241 , 0.0083904 ,
       0.00312214])

In [96]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def eval_with_threshold(model_name, probs, y_test, threshold=0.5):
    
    preds = (probs >= threshold).astype(int)
    
    metrics = {
        "model": f"{model_name}_thr_{threshold}",
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "f1_macro": f1_score(y_test, preds, average="macro", zero_division=0)
    }
    
    return metrics

In [99]:
probs = best_logreg.predict_proba(X_test)[:, 1]

results_threshold = []

for t in [0.20,0.25,0.3, 0.4, 0.5, 0.6]:
    results_threshold.append(
        eval_with_threshold("LogReg_Baseline", probs, y_test, threshold=t)
    )

threshold_df = pd.DataFrame(results_threshold).sort_values("recall", ascending=False)

threshold_df

,model,accuracy,precision,recall,f1,f1_macro
0,LogReg_Baseline_thr_0.2,0.716111,0.480422,0.852941,0.614644,0.694962
1,LogReg_Baseline_thr_0.25,0.747339,0.515517,0.799465,0.626834,0.717924
2,LogReg_Baseline_thr_0.3,0.760823,0.535238,0.751337,0.625139,0.724763
3,LogReg_Baseline_thr_0.4,0.787083,0.588095,0.660428,0.622166,0.736972
4,LogReg_Baseline_thr_0.5,0.802697,0.653846,0.545455,0.594752,0.732179
5,LogReg_Baseline_thr_0.6,0.796309,0.698630,0.409091,0.516020,0.693516


In [ ]:
# If you want higher Recall, you will usually pay with lower Precision.
# If you want higher Precision, you will usually lose Recall.

# Precision vs Recall in Churn Prediction (Simple Explanation)

## 1️ Recall = Customer Loss Risk

**Recall (for churn = 1)** answers this question:

> From all customers who will actually leave, how many did we catch?

- If **Recall is low** → many real churners were missed  
- That means **False Negatives are high**  
- Missed churners = 💸 **Revenue Loss**

So:

**Customer loss is related to False Negatives.**  
When Recall increases → missed churners decrease.

---

## 2️ Precision = Contact Cost

**Precision (for churn = 1)** answers this:

> From all customers we predicted as “will leave”, how many were correct?

- If **Precision is low** → many customers were contacted unnecessarily  
- That means **False Positives are high**  
- Wrong contacts = 📞 **Call / Offer / Discount Cost**

So:

**Contact cost is related to False Positives.**  
When Precision increases → wrong contacts decrease.

---

## 3️ What Does the Threshold Do?

The threshold controls how easily we label someone as “churn”.

---

### Lower Threshold (example: 0.5 → 0.3)

- More customers are classified as churn
- **Recall increases**
- **Precision decreases**

Result:
- 💸 Customer loss decreases  
- 📞 Contact cost increases  

---

### Higher Threshold (example: 0.5 → 0.6)

- Only strong cases are classified as churn
- **Precision increases**
- **Recall decreases**

Result:
- 📞 Contact cost decreases  
- 💸 Customer loss increases  

---

## 4️ The Golden Rule

> If you increase Recall, you usually lose Precision.  
> If you increase Precision, you usually lose Recall.

---

## 5️ Example Table

| Threshold | Recall | Precision | Business Effect |
|------------|---------|------------|----------------|
| 0.2 | 0.85 | 0.48 | Fewer lost customers, more wrong calls |
| 0.6 | 0.40 | 0.69 | Fewer wrong calls, more lost customers |

---

## 6️ How Do You Decide?

It depends on business strategy:

- If the company can afford more calls and discounts  
  → Choose lower threshold (focus on Recall)

- If the company wants to reduce operational cost  
  → Choose higher threshold (focus on Precision)
